# Entrenamiento YOLOv8s - Deteccion de danos en patrimonio (best.pt)

Notebook reconstruido a partir de los metadatos del checkpoint `best.pt`:

- Arquitectura: `yolov8s.pt` (11.1M parametros)
- Clases: `crack`, `humidity`, `spalling`
- Ultralytics 8.4.56 | 100 epocas | batch 16 | imgsz 640 | AdamW | seed 42
- Dataset original en `/content/datasets/unified_dataset.yaml`

Para reproducir este entrenamiento: sube tu dataset a Google Drive con la estructura que se indica abajo.

In [ ]:
# 1) Montar Google Drive y verificar GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Instalar la misma version de ultralytics usada para entrenar best.pt
!pip install -q ultralytics==8.4.56

# 3) Estructura del dataset (YOLO)

Copia tu dataset a `/content/datasets/` (o vincularlo desde Drive):

```
datasets/
├── unified_dataset.yaml
├── train/
│   ├── images/
│   │   ├── img_00001.jpg
│   │   └── ...
│   └── labels/
│       ├── img_00001.txt
│       └── ...
└── val/
    ├── images/
    └── labels/
```

`unified_dataset.yaml`:
```yaml
path: /content/datasets
train: train/images
val: val/images
names:
  0: crack
  1: humidity
  2: spalling
```

In [ ]:
# 4) ENTRENAMIENTO (reproduce los hiperparametros de best.pt)
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="/content/datasets/unified_dataset.yaml",  # ruta original en Colab
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    device=0,                # GPU
    workers=8,
    project="runs/detect",
    name="heritage_damage_v1",
    exist_ok=True,
    optimizer="AdamW",
    seed=42,
    deterministic=True,
    save_period=10,          # guarda runs/detect/heritage_damage_v1/weights/epoch*.pt
)

In [ ]:
# 5) Ver metricas finales del mejor modelo
from ultralytics.utils import RESULTS
best_pt = "runs/detect/heritage_damage_v1/weights/best.pt"

tuned = YOLO(best_pt)
m = tuned.val(data="/content/datasets/unified_dataset.yaml")
print(m.results_dict)

In [ ]:
# 6) Copiar best.pt al Drive y descargarlo
!cp runs/detect/heritage_damage_v1/weights/best.pt "/content/drive/MyDrive/best.pt"
from google.colab import files
files.download("runs/detect/heritage_damage_v1/weights/best.pt")